In [52]:
import pandas as pd
import sqlalchemy as db

import random

from pymongo.mongo_client import MongoClient
from pymongo.server_api import ServerApi

from dotenv import load_dotenv
import os

load_dotenv("../configuration/.env")

True

## read excel file

In [53]:
movies_df = pd.read_excel("../data/moviesCollection.xlsx",sheet_name='movie')
movies_df = movies_df[['title','gender','releaseDate','AwardMovie']]

movies_df.head()

                                               title  gender releaseDate  \
0     Trip to the Moon, A (Voyage dans la lune, Le)   Action  1902-01-01   
1                            Birth of a Nation, The    Drama  1915-01-01   
2  Intolerance: Love's Struggle Throughout the Ages    Drama  1916-01-01   
3                      20,000 Leagues Under the Sea   Action  1916-01-01   
4                                    Immigrant, The   Comedy  1917-01-01   

  AwardMovie  
0     Grammy  
1   Sin Info  
2      Oscar  
3      Oscar  
4   Sin Info  


## generate random score

In [54]:
def getRandomScore():
  if random.random() < 0.8:
    return random.randint(1,5)
  else:
    return None

movies_df["netflix_score"]=movies_df.apply(lambda _: getRandomScore(), axis=1)
movies_df["imdb_score"]=movies_df.apply(lambda _: getRandomScore(), axis=1)
movies_df["sensacine_score"]=movies_df.apply(lambda _: getRandomScore(), axis=1)
movies_df["rottentomatoes_score"]=movies_df.apply(lambda _: getRandomScore(), axis=1)

movies_df.head()

                                               title  gender releaseDate  \
0     Trip to the Moon, A (Voyage dans la lune, Le)   Action  1902-01-01   
1                            Birth of a Nation, The    Drama  1915-01-01   
2  Intolerance: Love's Struggle Throughout the Ages    Drama  1916-01-01   
3                      20,000 Leagues Under the Sea   Action  1916-01-01   
4                                    Immigrant, The   Comedy  1917-01-01   

  AwardMovie  netflix_score  imdb_score  sensacine_score  rottentomatoes_score  
0     Grammy            1.0         NaN              3.0                   4.0  
1   Sin Info            2.0         4.0              1.0                   5.0  
2      Oscar            5.0         NaN              4.0                   1.0  
3      Oscar            2.0         4.0              NaN                   3.0  
4   Sin Info            NaN         3.0              2.0                   2.0  


## generate user interactions


In [55]:
def generate_user_activity(movies_df, user_count):
    # Generate random user names
    users = [f"user_{i}" for i in range(user_count)]

    user_activities = []

    for row in range(movies_df.shape[0]):
        user = random.choice(users)
        activity = {
            "user": user,
            "movieTitle": movies_df.iloc[row]["title"],
            "finishCount": random.randint(1, 2) if random.randint(0,100)<70 else random.randint(0, 5),  # Random finish count between 0 and 4
            "backClickCount": random.randint(0, 2) if random.randint(0,100)<70 else random.randint(1, 10),  # Random back click count between 0 and 9
            "movieForwardCount": random.randint(0, 2) if random.randint(0,100)<70 else random.randint(1, 5),  # Random forward count between 0 and 4
            "playCount": random.randint(0, 3) if random.randint(0,100)<70 else random.randint(0, 10),  # Random play count between 1 and 9
            "movieViewPercentage": 100 if random.randint(0,100)<70 else round(random.uniform(0, 100), 2),  # Random view percentage between 0 and 100
        }
        user_activities.append(activity)

    return pd.DataFrame(user_activities)

In [56]:
user_interactions_df = generate_user_activity(movies_df,30)

user_interactions_df.head()

      user                                         movieTitle  finishCount  \
0   user_4     Trip to the Moon, A (Voyage dans la lune, Le)             1   
1  user_26                            Birth of a Nation, The             2   
2   user_7  Intolerance: Love's Struggle Throughout the Ages             2   
3  user_20                      20,000 Leagues Under the Sea             3   
4  user_17                                    Immigrant, The             2   

   backClickCount  movieForwardCount  playCount  movieViewPercentage  
0               2                  1          2               100.00  
1               1                  3          1                61.63  
2               8                  0          3               100.00  
3               2                  4          2                49.76  
4               1                  1          1               100.00  


## paste collections to MongoDB

In [57]:
# conexion a mongo
uri_Mongo = f"mongodb+srv://{os.getenv('MONGODB_USERNAME')}:{os.getenv('MONGODB_PASSWORD')}@{os.getenv('MONGODB_HOST')}?appname={os.getenv('MONGODB_CLUSTER_NAME')}"
clientMongo = MongoClient(uri_Mongo, server_api=ServerApi('1'))

c:\Users\joa_g\AppData\Local\Programs\Python\Python312\Lib\site-packages\pymongo\uri_parser.py:313: UserWarning: Unknown option: ?appname. Did you mean one of (appname, tz_aware, srvservicename) or maybe a camelCase version of one? Refer to docstring.
  return get_validated_options(opts, warn)


In [58]:
# create a schema:
dbMongo = clientMongo['netflix_movies']

# Create collections to store data:
interactions_collection =  dbMongo["user_netflix_interactions"]
movies_collection = dbMongo["movies_df"]

#paste data
interactions_collection.insert_many(user_interactions_df.to_dict('records'))
movies_collection.insert_many(movies_df.to_dict('records'))

InsertManyResult([ObjectId('66836bf21e67c1c81c3ca440'), ObjectId('66836bf21e67c1c81c3ca441'), ObjectId('66836bf21e67c1c81c3ca442'), ObjectId('66836bf21e67c1c81c3ca443'), ObjectId('66836bf21e67c1c81c3ca444'), ObjectId('66836bf21e67c1c81c3ca445'), ObjectId('66836bf21e67c1c81c3ca446'), ObjectId('66836bf21e67c1c81c3ca447'), ObjectId('66836bf21e67c1c81c3ca448'), ObjectId('66836bf21e67c1c81c3ca449'), ObjectId('66836bf21e67c1c81c3ca44a'), ObjectId('66836bf21e67c1c81c3ca44b'), ObjectId('66836bf21e67c1c81c3ca44c'), ObjectId('66836bf21e67c1c81c3ca44d'), ObjectId('66836bf21e67c1c81c3ca44e'), ObjectId('66836bf21e67c1c81c3ca44f'), ObjectId('66836bf21e67c1c81c3ca450'), ObjectId('66836bf21e67c1c81c3ca451'), ObjectId('66836bf21e67c1c81c3ca452'), ObjectId('66836bf21e67c1c81c3ca453'), ObjectId('66836bf21e67c1c81c3ca454'), ObjectId('66836bf21e67c1c81c3ca455'), ObjectId('66836bf21e67c1c81c3ca456'), ObjectId('66836bf21e67c1c81c3ca457'), ObjectId('66836bf21e67c1c81c3ca458'), ObjectId('66836bf21e67c1c81c3ca4